## 1. Get data

In [ ]:
import DBConn as dbc
import pandas as pd
import numpy as np
from pprint import pprint

In [ ]:
db = dbc.DBConn('config/silnaewichi-2c66f-firebase-adminsdk-t6ob2-a6dfec601d.json')
key = 'it5'

In [ ]:
bbsid_list = []
ssid_list = []

def getFrame(data:dict, x:float, y:float):
    global bbsid_list, ssid_list
    dict_frame = {
        'x': [x for _ in range(data['n'])],
        'y': [y for _ in range(data['n'])],
        'magnetic_x': [],
        'magnetic_y': [],
        'magnetic_z': []
    }
    
    mag_tag = "magnetic"
    wifi_tag = "wifi"
    bssid_tag = "BSSID"
    
    for _, obj in data.items():
        if _ == 'n':
            continue
        # magnetic
        for dim, d in obj[mag_tag].items():
            dict_frame[f"{mag_tag}_{dim}"].append(d)
        # wifi list setting
        for _, wifi in obj[wifi_tag].items():
            if wifi['BSSID'] not in bbsid_list:
                bbsid_list.append(wifi[bssid_tag])
                ssid_list.append(wifi['SSID'])
                
    # wifi
    for bbsid in bbsid_list:
        dict_frame[bbsid] = []
    for _, obj in data.items():
        if _ == 'n':
            continue
        none_flag = {bb:False for bb in bbsid_list}
        for _, wifi in obj[wifi_tag].items():
            dict_frame[wifi[bssid_tag]].append(wifi['WifiLevel'])
            none_flag[wifi[bssid_tag]] = True
        for bb, right in none_flag.items():
            if right is False:
                dict_frame[bb].append(None)
            
    result_df = pd.DataFrame(data=dict_frame, dtype=np.float64)
    return result_df


* x: 0 ~ 12
* y:-8 ~ 8
* x break points : 0, 10, 12
* y break points : 0, 3, 8

In [ ]:
%%time
all_f = []

for y in [0, 3, 8]:
    for x in range(1, 10):
        _d = db.getData(key, x, y)
        _f = getFrame(_d, x, y)
        all_f.append(_f)
        
for x in [0, 10]:
    for y in range(0, 9):
        _d = db.getData(key, x, y)
        _f = getFrame(_d, x, y)
        all_f.append(_f)
        
y = 0
for x in [11, 12]:
    _d = db.getData(key, x, y)
    _f = getFrame(_d, x, y)
    all_f.append(_f)
    
x = 12
for y in range(-8, 0):
    _d = db.getData(key, x, y)
    _f = getFrame(_d, x, y)
    all_f.append(_f)

In [ ]:
dict_frame = {
    'x': [],
    'y': [],
    'magnetic_x': [],
    'magnetic_y': [],
    'magnetic_z': []
}
for bss in bbsid_list:
    dict_frame[bss] = []
    
result_df = pd.DataFrame(data=dict_frame)
for _df in all_f:
    result_df = result_df.append(_df, ignore_index=True)
result_df.head()
result_df.shape

In [ ]:
result_df.dtypes

In [ ]:
result_df.isnull().sum()

In [ ]:
result_df.to_csv("./data/full_data.csv")

In [ ]:
bssid_ssid = dict()
for i, (b, n) in enumerate(zip(bbsid_list, ssid_list)):
    bssid_ssid[b] = n
pprint(bssid_ssid)

## 2. data cleansing & fillna

In [ ]:
cut_df = result_df.copy()
cut_df.shape

In [ ]:
result_df.isnull().sum() > 30

In [ ]:
drop_list = [
    '68:ca:e4:46:c1:80',
    '68:ca:e4:46:c1:81',
    '68:ca:e4:47:4d:e0',
    '68:ca:e4:47:4d:e1',
    '68:ca:e4:47:4d:ee',
    '68:ca:e4:47:4d:ef',
    '68:ca:e4:47:4e:a0',
    '68:ca:e4:47:4e:a1',
    '68:ca:e4:47:54:21',
    '68:ca:e4:47:6e:e0',
    '68:ca:e4:48:9d:a0',
    '68:ca:e4:48:9d:a1',
    '68:ca:e4:48:9d:ae',
    '68:ca:e4:48:9d:af',
    '88:36:6c:08:80:8c',
    '88:36:6c:08:80:8e',
    '88:36:6c:a1:47:7a',
    '88:36:6c:bb:10:06',
    '90:9f:33:a8:6b:b4',
    '08:d4:6a:e6:ff:39' # Q8
]
print("will dropped:")
for dd in drop_list:
    print(bssid_ssid[dd])

In [ ]:
fin_df = cut_df.drop(drop_list, axis=1)
fin_df.shape, fin_df.columns

In [ ]:
print("survived wifi list")
for bb in (set(bbsid_list) - set(drop_list)):
    print(f"{bb}: {bssid_ssid[bb]}")

In [ ]:
fin_df.fillna(-100.0, inplace=True)

In [ ]:
fin_df.isnull().sum()

In [ ]:
fin_df.to_csv("./data/cut_off_data.csv")